In [2]:
import os
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_ollama import ChatOllama
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, RemoveMessage
from typing import Literal
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
api_key = os.getenv("API_KEY")

In [4]:
chat = ChatOllama(
    model="gpt-oss:120b",
    temperature=0.7,
    base_url="https://ollama.com",
    client_kwargs={
        "headers": {
            "Authorization": f"Bearer {api_key}"
        }
    }
)

In [5]:
class State(MessagesState):
    summary: str

In [6]:
test_state = State()

In [9]:
# test_state.get("summary","") # Returns "" if nothing is available in summary
bool(test_state.get("summary",""))

False

In [10]:
def ask_question(state: State) -> State:
    
    print(f"\n-------> ENTERING ask_question:")
    
    question = "What is your question?"
    print(question)
    
    return State(messages = [AIMessage(question), HumanMessage(input())])

In [12]:
def chatbot(state: State) -> State:
    
    print(f"\n-------> ENTERING chatbot:")
    for i in state["messages"]:
        i.pretty_print()
        
    system_message = f'''
    Here's a quick summary of what's been discussed so far:
    {state.get("summary", "")}
    
    Keep this in mind as you answer the next question.
    '''
    
    response = chat.invoke([SystemMessage(system_message)] + state["messages"])
    response.pretty_print()
    
    return State(messages = [response])

In [13]:
def ask_another_question(state: State) -> State:
    
    print(f"\n-------> ENTERING ask_another_question:")
    
    question = "Would you like to ask one more question (yes/no)?"
    print(question)
    
    return State(messages = [AIMessage(question), HumanMessage(input())])

In [14]:
def summarize_and_delete_messages(state: State) -> State:
    print(f"\n-------> ENTERING trim_messages:")
    
    new_conversation = ""
    for i in state["messages"]:
        new_conversation += f"{i.type}: {i.content}\n\n"
        
    summary_instructions = f'''
Update the ongoing summary by incorporating the new lines of conversation below.  
Build upon the previous summary rather than repeating it so that the result  
reflects the most recent context and developments.


Previous Summary:
{state.get("summary", "")}

New Conversation:
{new_conversation}
'''
    
    print(summary_instructions)
    
    summary = chat.invoke([HumanMessage(summary_instructions)])
    
    remove_messages = [RemoveMessage(id = i.id) for i in state["messages"][:]]
    
    return State(messages = remove_messages, summary = summary.content)

In [15]:
def routing_function(state: State) -> Literal["summarize_and_delete_messages", "__end__"]:
    
    if state["messages"][-1].content == "yes":
        return "summarize_and_delete_messages"
    else:
        return "__end__"

In [16]:
graph = StateGraph(State)

In [17]:
graph.add_node("ask_question", ask_question)
graph.add_node("chatbot", chatbot)
graph.add_node("ask_another_question", ask_another_question)
graph.add_node("summarize_and_delete_messages", summarize_and_delete_messages)

graph.add_edge(START, "ask_question")
graph.add_edge("ask_question", "chatbot")
graph.add_edge("chatbot", "ask_another_question")
graph.add_conditional_edges(source = "ask_another_question", 
                            path = routing_function)
graph.add_edge("summarize_and_delete_messages", "ask_question")

In [18]:
graph_compiled = graph.compile()

In [20]:
graph_compiled.invoke(State(messages = []))


-------> ENTERING ask_question:
What is your question?

-------> ENTERING chatbot:
================================== Ai Message ==================================

What is your question?
================================ Human Message =================================

describe javed akhtar in 50 words
================================== Ai Message ==================================

Javed Akhtar, born 1945, is a celebrated Indian poet, lyricist, and screenwriter. Renowned for his eloquent Urdu verses, he co‑founded the band India‑Shakti, won multiple Filmfare and National Awards, and champions secularism, gender equality, and literary freedom. His works blend wit, humanity, and timeless resonance in contemporary Indian cinema today.

-------> ENTERING ask_another_question:
Would you like to ask one more question (yes/no)?

-------> ENTERING trim_messages:

Update the ongoing summary by incorporating the new lines of conversation below.  
Build upon the previous summary rather than rep

{'messages': [AIMessage(content='What is your question?', additional_kwargs={}, response_metadata={}, id='22232ab8-b65c-4b98-9687-97b2c88da393', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='Name of his kids', additional_kwargs={}, response_metadata={}, id='792a0080-ecd2-4932-a182-04d894ec322d'),
  AIMessage(content='Javed\u202fAkhtar has two children:\n\n1. **Farhan\u202fAkhtar** – his son, a well‑known actor, director, singer‑songwriter and film producer.  \n2. **Zoya\u202fAkhtar** – his daughter, a celebrated film director and screen‑writer.  \n\nBoth Farhan and Zoya are the children he shares with his wife, actress\u202fShabana\u202fAzmi.', additional_kwargs={}, response_metadata={'model': 'gpt-oss:120b', 'created_at': '2026-06-29T18:24:46.298945631Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10504636509, 'load_duration': None, 'prompt_eval_count': 205, 'prompt_eval_duration': None, 'eval_count': 975, 'eval_duration': None, 'logprobs': None, 'model_na